# Clase 6 — Optimiza tus modelos: búsqueda de hiperparámetros

**Manual asociado:** [manual/06_optimiza_hiperparametros.md](../manual/06_optimiza_hiperparametros.md)

## Resumen para la clase

- Los hiperparámetros son decisiones de diseño que fijas antes de entrenar.
- Un bucle que genera y entrena combinaciones te permite comparar arquitecturas de forma objetiva.
- Grid Search, Random Search y Bayesian Optimization son alternativas más eficientes que probar a mano.
- Dropout, BatchNormalization y early stopping complementan la búsqueda de arquitectura.

In [ ]:
%load_ext tensorboard

In [ ]:
import pickle
with open("X.pickle", "rb") as archivo:
    X = pickle.load(archivo)
with open("y.pickle", "rb") as archivo:
    y = pickle.load(archivo)
X = X / 255.0

In [ ]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import TensorBoard

# Version reducida para que la clase entrene en poco tiempo; en tu practica personal
# puedes ampliar estas listas
capas_conv = [1, 3]
tamanos_capa = [32, 64]
capas_densas = [0, 1]

for num_capas_conv in capas_conv:
    for tamano in tamanos_capa:
        for num_capas_densas in capas_densas:
            nombre_modelo = f"{num_capas_conv}conv-{tamano}nodos-{num_capas_densas}densas-{int(time.time())}"
            tensorboard_callback = TensorBoard(log_dir=f"logs/{nombre_modelo}")
            print(f"Entrenando: {nombre_modelo}")

            model = Sequential()
            model.add(Conv2D(tamano, (3, 3), input_shape=X.shape[1:]))
            model.add(Activation("relu"))
            model.add(MaxPooling2D(pool_size=(2, 2)))

            for _ in range(num_capas_conv - 1):
                model.add(Conv2D(tamano, (3, 3)))
                model.add(Activation("relu"))
                model.add(MaxPooling2D(pool_size=(2, 2)))

            model.add(Flatten())

            for _ in range(num_capas_densas):
                model.add(Dense(tamano))
                model.add(Activation("relu"))

            model.add(Dense(1, activation="sigmoid"))

            model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
            model.fit(
                X, y,
                batch_size=32,
                epochs=5,
                validation_split=0.1,
                callbacks=[tensorboard_callback],
                verbose=0,
            )

In [ ]:
%tensorboard --logdir logs

## Cierre de la clase

- Compara en TensorBoard el `val_accuracy` final de cada combinación: el nombre del modelo te dice qué hiperparámetros usó.
- En la clase 7 usas el mejor modelo entrenado hasta ahora para predecir sobre imágenes nuevas.